In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib
# matplotlib.use('agg')
import matplotlib.pyplot as plt
import os
# from tqdm.notebook import  tqdm
from tqdm import  tqdm
import talib
import datetime
import math
import  mplfinance as mpf

import sys
sys.path.append('../../DataSource/baostock')
import datasource
sys.path.append('../..')
import Utils
# 这个是筛选多少天上涨多少的，
codes = datasource.get_codes()
code = codes[0]
dt = datasource.get_data(code)
dt.head()

,code,open,high,low,close,preclose,volume,amount,adjustflag,turn,tradestatus,pctChg,isST
date,,,,,,,,,,,,,
2010-01-04,sh.600000,5.079055,5.088362,4.923170,4.930150,5.046482,66191338,1.419984e+09,2,0.835129,1,-2.3052,0
2010-01-05,sh.600000,4.981336,5.020889,4.837085,4.967376,4.930150,115147943,2.436891e+09,2,1.452808,1,0.7551,0
2010-01-06,sh.600000,4.953417,4.955743,4.858024,4.869658,4.967376,96782575,2.034174e+09,2,1.221095,1,-1.9672,0
2010-01-07,sh.600000,4.858024,4.895251,4.723079,4.760305,4.869658,85236072,1.761801e+09,2,1.075414,1,-2.2456,0
2010-01-08,sh.600000,4.732386,4.839411,4.723079,4.813818,4.760305,65707646,1.349532e+09,2,0.829026,1,1.1241,0


In [7]:
DAYS = 3 # 连续多日
AFTER = 5 # 后边多少天的
for i in range(DAYS):
    dt[f'preclose{i+1}'] = dt['close'].shift(i+1) # 往下移动
    dt[f'turn{i+1}'] = dt['turn'].shift(i+1)      # 同样是往下移动

for i in range(AFTER):
    dt[f'nextclose{i+1}'] = dt['close'].shift(-(i+1)) # 往上移动
dt2 = dt.loc[(dt['close'] > dt['preclose1']) & (dt['preclose1'] > dt['preclose2']) & (dt['preclose2'] > dt['preclose3']) & (dt['turn'] >dt['turn1']) & (dt['turn1'] > dt['turn2']) & (dt['turn2'] > dt['turn3'])]
print(len(dt2))
dt2.tail()

42


,code,open,high,low,close,preclose,volume,amount,adjustflag,turn,...,nextclose1,nextclose2,nextclose3,nextclose4,nextclose5,rate1,rate2,rate3,rate4,rate5
date,,,,,,,,,,,,,,,,,,,,,
2024-05-20,sh.600000,7.676930,7.817362,7.630120,7.789276,7.639482,88144991,7.311044e+08,2,0.3003,...,7.882897,7.985880,7.976518,7.901621,7.995242,1.012019,1.025240,1.024038,1.014423,1.026442
2025-04-16,sh.600000,10.142425,10.288010,10.045368,10.288010,10.161836,74094520,7.773786e+08,2,0.2524,...,10.433595,10.491829,10.258893,10.394773,10.355950,1.014151,1.019811,0.997170,1.010377,1.006604
2025-04-30,sh.600000,10.501535,10.705354,10.288010,10.637414,10.511241,92838022,1.011036e+09,2,0.3163,...,10.841233,11.035347,11.287694,11.452691,11.365340,1.019161,1.037409,1.061131,1.076642,1.068431
2025-05-06,sh.600000,10.676237,10.860645,10.482124,10.841233,10.637414,97264620,1.076041e+09,2,0.3314,...,11.035347,11.287694,11.452691,11.365340,11.724449,1.017905,1.041182,1.056401,1.048344,1.081468
2025-08-05,sh.600000,13.150000,13.780000,13.140000,13.750000,13.130000,133646594,1.812400e+09,2,0.4415,...,13.790000,13.930000,14.170000,13.980000,13.890000,1.002909,1.013091,1.030545,1.016727,1.010182


In [8]:
# 然后这里看结果
for i in range(AFTER):
    dt[f'rate{i+1}'] = dt[f'nextclose{i+1}']/dt['close']
    _mean = dt[f'rate{i+1}'].mean()
    _median = dt[f'rate{i+1}'].median()
    _std = dt[f'rate{i+1}'].std()
    print(f'{i+1}天，均值:{_mean}，中位值:{_median}，标准差:{_std}')

1天，均值:1.0003680800416794，中位值:1.0，标准差:0.015958548186219813
2天，均值:1.0007320940658329，中位值:1.0，标准差:0.022558927832483103
3天，均值:1.0010972048771054，中位值:1.0，标准差:0.027668989521742893
4天，均值:1.0014620573609156，中位值:1.0，标准差:0.032138506871572825
5天，均值:1.001828604573585，中位值:1.0，标准差:0.03618390934981988


In [ ]:
# 然后我要看看，如果